# Week 8 — Functions

**Notebook outcomes**

- Define functions with `def` and return values
- Use positional, keyword, default, `*args`, and `**kwargs` arguments
- Add type hints and docstrings
- Reason about scope and the difference between pure and side-effecting functions
- Compose and informally test small functions


## Why functions?

Three reasons, in priority order:

1. **Naming.** A well-named function lets you forget the details and
   read the surrounding code at the right level of abstraction.
2. **Reuse.** You write the logic once, call it from many places.
3. **Testing.** Pure functions are the easiest kind of code to test.

"If you copy-paste a block of code twice, make it a function" is close
to a law.


## Defining a function

```python
def name(parameters):
    """Optional docstring."""
    body
    return value
```


In [ ]:
def square(x):
    """Return x squared."""
    return x * x


print(square(5))
print(square(1.5))

### `return`

- A function returns a value when it hits `return`.
- Without a `return`, it returns `None`.
- You can return **multiple** values as a tuple (implicitly):


In [ ]:
def min_max(numbers):
    return min(numbers), max(numbers)


lo, hi = min_max([3, 1, 4, 1, 5, 9])
print(lo, hi)

### Docstrings

The first string inside a function is its **docstring**. It describes
what the function does. Keep the first line short and imperative.

```python
def price(shares, cost):
    """Return total cost of `shares` at `cost` per share."""
    return shares * cost
```

`help(price)` (or hovering in most editors) shows this docstring.


## Arguments

### Positional arguments

The most common. Order matters.


In [ ]:
def wage(hours, rate):
    return hours * rate


print(wage(40, 25.0))

### Keyword arguments

You can pass arguments by **name**. Order no longer matters, and the
call site is more self-documenting.


In [ ]:
print(wage(rate=25.0, hours=40))

### Default values

Parameters can have default values. Defaults are evaluated **once**, at
function definition time — which causes a famous Python trap (below).


In [ ]:
def greet(name, greeting="Hello"):
    return f"{greeting}, {name}!"


print(greet("Chase"))
print(greet("Chase", "Bonjour"))
print(greet("Chase", greeting="Howdy"))

### The mutable-default trap

Don't use mutable values (lists, dicts, sets) as defaults. This shocks
newcomers.


In [ ]:
def append_item(item, bucket=[]):  # DANGEROUS
    bucket.append(item)
    return bucket


print(append_item(1))  # [1]        ... ok
print(append_item(2))  # [1, 2]     # wait, what?
print(append_item(3))  # [1, 2, 3]  # same list, every call

The fix is to use `None` as the default and create a fresh object
inside the function:


In [ ]:
def append_item(item, bucket=None):
    if bucket is None:
        bucket = []
    bucket.append(item)
    return bucket


print(append_item(1))
print(append_item(2))
print(append_item(3))

### `*args` and `**kwargs`

When you don't know how many arguments the caller will pass:


In [ ]:
def mean(*values):  # *args
    return sum(values) / len(values)


print(mean(1, 2, 3, 4, 5))

In [ ]:
def config(**options):  # **kwargs
    for key, value in options.items():
        print(f"{key} = {value}")


config(alpha=0.5, beta=0.99, n=100)

A full signature can mix positional, default, `*args`, and
`**kwargs`:

```python
def f(a, b, *args, c=10, **kwargs):
    ...
```

Rare to need all four. Know they exist so library code reads cleanly.


## Type hints

Type hints are annotations that say what types the function expects and
returns. Python doesn't check them at runtime (by default), but editors
and tools like `ruff`/`mypy` do — and they're great documentation.

```python
def wage(hours: float, rate: float) -> float:
    return hours * rate
```


In [ ]:
def wage(hours: float, rate: float) -> float:
    return hours * rate


print(wage(40.0, 25.0))

Common hints:

```python
from collections.abc import Iterable

def mean(values: list[float]) -> float: ...
def names_to_gpa(records: dict[str, float]) -> list[str]: ...
def maybe_parse(text: str) -> int | None: ...
def apply(fn: callable, values: Iterable[float]) -> list[float]: ...
```

The `|` union syntax (`int | None`) works on Python 3.10+.


## Scope

A variable inside a function is **local** to that call. It doesn't leak
out.


In [ ]:
def f():
    x = 1
    print("inside:", x)


f()
# print(x)   # NameError: x is not defined out here

Python looks up names in **LEGB** order: **L**ocal → **E**nclosing
→ **G**lobal → **B**uilt-in. If you read a variable that's not local,
Python checks the enclosing function(s), then the module, then the
built-ins.

Don't assign to a global inside a function. If you find yourself needing
to, refactor — probably that global wants to be a parameter or a
returned value.


## Pure vs. side-effecting functions

A function is **pure** when:

1. It returns the same output for the same inputs.
2. It doesn't mutate its arguments or any external state.

Pure functions are easier to reason about, test, and reuse. Favor them.

When you *must* mutate something — writing a file, printing, or
modifying a list — keep those functions small and don't mix them with
calculation-heavy code.


In [ ]:
# Pure — recommended
def yoy_growth(prev: float, curr: float) -> float:
    """Return year-over-year growth rate as a decimal."""
    return (curr - prev) / prev


# Side-effecting — prints instead of returning
def print_growth(prev: float, curr: float) -> None:
    rate = (curr - prev) / prev
    print(f"growth: {rate:.2%}")


# A good pattern: separate the calculation from the output
def report_growth(prev: float, curr: float) -> None:
    rate = yoy_growth(prev, curr)
    print(f"growth: {rate:.2%}")

## Composing small functions

Build up from small pieces. Each function does **one thing**.


In [ ]:
from collections.abc import Iterable


def mean(values: Iterable[float]) -> float:
    values = list(values)
    return sum(values) / len(values)


def demean(values: list[float]) -> list[float]:
    m = mean(values)
    return [v - m for v in values]


def variance(values: list[float]) -> float:
    d = demean(values)
    return sum(v * v for v in d) / len(d)


xs = [1.0, 2.0, 3.0, 4.0, 5.0]
print("mean:", mean(xs))
print("demean:", demean(xs))
print("variance:", variance(xs))

## Informal testing

A quick way to check your function: `assert` the expected result for a
small case. If the assertion fails, Python raises `AssertionError` and
you know the function is broken.


In [ ]:
def yoy_growth(prev, curr):
    return (curr - prev) / prev


assert yoy_growth(100, 110) == 0.10
assert yoy_growth(200, 100) == -0.50
print("all good")

In week 11 we'll replace these ad-hoc asserts with `pytest`, a
proper testing framework. The idea is the same.


## Recap

- `def` + docstring + body + `return`.
- Positional, keyword, default, `*args`, `**kwargs` — know the shape.
- Don't use mutable defaults. Use `None`.
- Type hints are documentation + editor help — free upside.
- Prefer **pure** functions; isolate side effects.
- Compose small functions. Test with `assert` for now.
